# Explorative Datenanalyse: S&P-500-Fundamentaldaten

Erster Blick auf `data/raw/fundamentals.csv`, bevor die Daten in `src/clean.py` bereinigt werden.

Leitfragen:
1. Wie viele Zeilen/Spalten hat die Tabelle überhaupt?
2. Wie heißen die Spalten genau?
3. Gibt es fehlende Werte?
4. Sehen die Zahlen plausibel aus?

In [ ]:
from pathlib import Path

import pandas as pd

# Projektwurzel finden (der Ordner mit pyproject.toml), damit das Notebook
# unabhängig vom aktuellen Arbeitsverzeichnis funktioniert
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw" / "fundamentals.csv"

# Anzeigeoptionen: mehr Zeilen zeigen und große Zahlen ohne wissenschaftliche Notation
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

df = pd.read_csv(RAW)

## 1. Wie viele Zeilen/Spalten hat die Tabelle?

In [ ]:
# shape liefert (Anzahl Zeilen, Anzahl Spalten)
n_rows, n_cols = df.shape
print(f"Zeilen:  {n_rows:,}")
print(f"Spalten: {n_cols}")

# Eine Zeile entspricht einem Jahresabschluss eines Unternehmens.
# Daher interessiert auch: wie viele Unternehmen und wie viele Abschlüsse je Unternehmen?
print(f"Unternehmen (eindeutige Ticker): {df['Ticker Symbol'].nunique()}")
print("\nAbschlüsse je Unternehmen -> Anzahl Unternehmen:")
print(df["Ticker Symbol"].value_counts().value_counts().sort_index().to_string())

In [ ]:
# Die ersten Zeilen geben ein Gefühl für den Aufbau der Daten
df.head()

## 2. Wie heißen die Spalten genau?

In [ ]:
# Übersicht aller Spalten mit Datentyp und einem Beispielwert aus der ersten Zeile.
# Die Namen enthalten Leerzeichen und Sonderzeichen (z.B. "Sales, General and Admin."),
# deshalb werden sie in clean.py später in snake_case umbenannt.
columns = pd.DataFrame({
    "spalte": df.columns,
    "datentyp": df.dtypes.astype(str).values,
    "beispielwert": df.iloc[0].values,
})
columns

In [ ]:
# "Unnamed: 0" ist nur der mitexportierte Index der Originaldatei (0, 1, 2, ...) und kein echtes Merkmal
print("Unnamed: 0 entspricht der Zeilennummer:", (df["Unnamed: 0"] == range(len(df))).all())

# "Period Ending" wurde als Text eingelesen -> muss später in ein Datum umgewandelt werden
print("Datentyp von Period Ending:", df["Period Ending"].dtype)

## 3. Gibt es fehlende Werte?

In [ ]:
# Fehlende Werte je Spalte, absolut und in Prozent; nur Spalten mit Lücken anzeigen
missing = pd.DataFrame({
    "fehlend": df.isna().sum(),
    "fehlend_pct": df.isna().mean() * 100,
})
missing[missing["fehlend"] > 0].sort_values("fehlend", ascending=False)

In [ ]:
# Bei welchen Unternehmen fehlt die Current Ratio? Vermutung: Banken/Versicherer,
# die keine Trennung in kurz- und langfristige Bilanzposten ausweisen.
df.loc[df["Current Ratio"].isna(), "Ticker Symbol"].unique()[:20]

In [ ]:
# Neben echten Lücken (NaN) gibt es versteckte Probleme wie Duplikate.
# Exakte Duplikate (komplett identische Zeilen):
print("Exakte Duplikate:", df.duplicated().sum())

# Mehrere Abschlüsse für dasselbe Unternehmen im selben "For Year"
# -> Hinweis darauf, dass die Spalte "For Year" nicht zuverlässig ist
dup_year = df[df["For Year"].notna() & df.duplicated(["Ticker Symbol", "For Year"], keep=False)]
print("Zeilen mit doppelter Ticker/For-Year-Kombination:", len(dup_year))
dup_year[["Ticker Symbol", "Period Ending", "For Year", "Total Revenue"]].head(6)

## 4. Sehen die Zahlen plausibel aus?

In [ ]:
# Lageparameter der Kernspalten in Milliarden USD, damit die Größenordnung lesbar ist
key_cols = ["Total Revenue", "Cost of Revenue", "Gross Profit", "Net Income", "Total Assets"]
(df[key_cols] / 1e9).describe().T

In [ ]:
# Stichprobe mit bekannten Zahlen: Apples Umsatz lag im Geschäftsjahr 2015 bei rund 234 Mrd. USD
df.loc[df["Ticker Symbol"] == "AAPL", ["Period Ending", "Total Revenue", "Net Income"]]

In [ ]:
# Plausibilitätsregeln: jede Prüfung zählt die Zeilen, die gegen eine Erwartung verstoßen
checks = {
    "Umsatz <= 0": (df["Total Revenue"] <= 0).sum(),
    "Kosten > Umsatz": (df["Cost of Revenue"] > df["Total Revenue"]).sum(),
    # Bruttogewinn sollte Umsatz minus Herstellkosten sein (Toleranz 1.000 USD für Rundung)
    "Bruttogewinn != Umsatz - Kosten": (
        (df["Total Revenue"] - df["Cost of Revenue"] - df["Gross Profit"]).abs() > 1e3
    ).sum(),
    # Banken/Versicherer haben keine Herstellkosten -> Bruttomarge dort nicht aussagekräftig
    "Cost of Revenue = 0": (df["Cost of Revenue"] == 0).sum(),
    "Gross Margin außerhalb 0-100 %": (~df["Gross Margin"].between(0, 100)).sum(),
}
pd.Series(checks, name="anzahl_zeilen").to_frame()

In [ ]:
# Zeitliche Plausibilität: Der Datensatz soll etwa die Jahre 2012-2016 abdecken
period_year = pd.to_datetime(df["Period Ending"]).dt.year
print("Jahr des Periodenendes:")
print(period_year.value_counts().sort_index().to_string())

print("\nWerte in 'For Year' (NaN = fehlend):")
print(df["For Year"].value_counts(dropna=False).sort_index().to_string())

In [ ]:
# Ausreißer genauer ansehen: Abschlüsse vor 2010 und das unmögliche Jahr 1215
df.loc[(period_year < 2010) | (df["For Year"] == 1215),
       ["Ticker Symbol", "Period Ending", "For Year", "Total Revenue"]]

## Fazit

- **Größe:** 1.781 Zeilen × 79 Spalten, 448 Unternehmen, meist 4 Geschäftsjahre je Unternehmen.
- **Spalten:** 76 numerische Kennzahlen, dazu Ticker und Periodenende (als Text). `Unnamed: 0` ist ein überflüssiger Index.
- **Fehlende Werte:** Liquiditätskennzahlen (Cash/Current/Quick Ratio) fehlen v. a. bei Finanzunternehmen, außerdem EPS und Aktienanzahl. Diese Lücken sind fachlich erklärbar und werden nicht geschätzt.
- **Plausibilität:** Die Größenordnungen stimmen (z. B. Apple). Aber:
  - `For Year` ist lückenhaft, enthält den Fehlwert 1215 und erzeugt Duplikate → das Geschäftsjahr wird in `clean.py` aus `Period Ending` abgeleitet.
  - COTY hat einzelne Abschlüsse aus 2003–2007 mit unplausiblen Umsätzen → werden entfernt.
  - Banken/Versicherer haben `Cost of Revenue = 0` → Kostenquote und Bruttomarge werden in `transform.py` als „nicht anwendbar“ behandelt.